# Ejercicio 9: Uso de la API de Google Gemini con Retrieval

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Instalación de dependencias e importaciones

In [1]:
# Instalar librerías necesarias
!pip install google-generativeai scikit-learn sentence-transformers -q

print("✓ Instalación completada")

✓ Instalación completada


In [2]:
# Importar librerías
import google.generativeai as genai
from google.colab import userdata
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ Librerías importadas correctamente


In [4]:
# Obtener la API key del gestor de secretos de Colab
try:
    API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=API_KEY)
    print("✓ API Key configurada correctamente desde Colab Secrets")
except:
    print("No se encontró la API Key en Colab Secrets")

✓ API Key configurada correctamente desde Colab Secrets


## 2. Prueba básica de conexión

In [5]:
# Probar conexión listando modelos disponibles
print("Modelos disponibles:")
print("="*50)
try:
    for model in genai.list_models():
        capabilities = "supports_content_generation" if model.supported_generation_methods else ""
        print(f"  {model.display_name}")
    print("\nConexión a API exitosa")
except Exception as e:
    print(f"Error en conexión: {e}")
    print("Verifica que tu API Key sea correcta en Colab Secrets")

Modelos disponibles:
  ✓ Gemini 2.5 Flash
  ✓ Gemini 2.5 Pro
  ✓ Gemini 2.0 Flash
  ✓ Gemini 2.0 Flash 001
  ✓ Gemini 2.0 Flash-Lite 001
  ✓ Gemini 2.0 Flash-Lite
  ✓ Gemini 2.5 Flash Preview TTS
  ✓ Gemini 2.5 Pro Preview TTS
  ✓ Gemma 4 26B A4B IT
  ✓ Gemma 4 31B IT
  ✓ Gemini Flash Latest
  ✓ Gemini Flash-Lite Latest
  ✓ Gemini Pro Latest
  ✓ Gemini 2.5 Flash-Lite
  ✓ Nano Banana
  ✓ Gemini 3 Pro Preview
  ✓ Gemini 3 Flash Preview
  ✓ Gemini 3.1 Pro Preview
  ✓ Gemini 3.1 Pro Preview Custom Tools
  ✓ Gemini 3.1 Flash Lite Preview
  ✓ Gemini 3.1 Flash Lite
  ✓ Nano Banana Pro
  ✓ Nano Banana Pro
  ✓ Nano Banana Pro
  ✓ Nano Banana 2
  ✓ Nano Banana 2
  ✓ Gemini 3.5 Flash
  ✓ Lyria 3 Clip Preview
  ✓ Lyria 3 Pro Preview
  ✓ Gemini 3.1 Flash TTS Preview
  ✓ Gemini Robotics-ER 1.5 Preview
  ✓ Gemini Robotics-ER 1.6 Preview
  ✓ Gemini 2.5 Computer Use Preview 10-2025
  ✓ Antigravity Agent Preview
  ✓ Deep Research Max Preview (Apr-21-2026)
  ✓ Deep Research Preview (Apr-21-2026)
  ✓ Deep

## 3. Cargar el corpus de 20 News Groups

In [8]:
# Descargar dataset 20 News Groups
print("Descargando dataset 20 News Groups...")
print("Esto puede tomar un momento...\n")

try:
    newsgroups = fetch_20newsgroups(
        subset='train',
        categories=['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med'],
        remove=('headers', 'footers', 'quotes')
    )

    # Tomar solo los primeros 100 documentos para que sea más rápido
    documents = newsgroups.data[:100]

    print(f"Dataset cargado exitosamente")
    print(f"  - Total de documentos: {len(documents)}")
    print(f"\nPrimer documento (primeros 300 caracteres):")
    print(f"{'-'*80}")
    print(documents[0][:300] + "...")
except Exception as e:
    print(f"Error al descargar dataset: {e}")

Descargando dataset 20 News Groups...
Esto puede tomar un momento...

✓ Dataset cargado exitosamente
  - Total de documentos: 100

Primer documento (primeros 300 caracteres):
--------------------------------------------------------------------------------
Does anyone know of a good way (standard PC application/PD utility) to
convert tif/img/tga files into LaserJet III format.  We would also like to
do the same, converting to HPGL (HP plotter) files.

Please email any response.

Is this the correct group?

Thanks in advance.  Michael....


## 4. Generar embeddings de los documentos

In [9]:
# Cargar modelo de embeddings (ligero y rápido para Colab)
print("Cargando modelo de embeddings...")
print("Este modelo mapea texto a vectores numéricos\n")

try:
    embedding_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
    print("Modelo cargado")
except Exception as e:
    print(f"Error: {e}")

Cargando modelo de embeddings...
Este modelo mapea texto a vectores numéricos



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Modelo cargado


In [10]:
# Generar embeddings para todos los documentos
print("Generando embeddings para los documentos...")
print("Esto transforma cada documento en un vector matemático\n")

try:
    embeddings = embedding_model.encode(documents, show_progress_bar=True)

    print(f"\nEmbeddings generados")
    print(f"  - Número de documentos: {embeddings.shape[0]}")
    print(f"  - Dimensión de cada embedding: {embeddings.shape[1]}")
    print(f"\nEach documento está representado como un vector de {embeddings.shape[1]} dimensiones")
except Exception as e:
    print(f"Error: {e}")

Generando embeddings para los documentos...
Esto transforma cada documento en un vector matemático



Batches:   0%|          | 0/4 [00:00<?, ?it/s]


✓ Embeddings generados
  - Número de documentos: 100
  - Dimensión de cada embedding: 384

Each documento está representado como un vector de 384 dimensiones


## 5. Búsqueda de documentos similares (Retrieval)

In [11]:
# Crear una query y buscar documentos similares
query = "¿Cuáles son los tratamientos para enfermedades del corazón?"

print(f"Query: {query}")
print(f"{'-'*80}\n")

try:
    # Generar embedding para la query
    query_embedding = embedding_model.encode([query])

    # Calcular similaridad del coseno entre la query y todos los documentos
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Obtener los índices de los 5 documentos más similares
    top_5_indices = np.argsort(similarities)[-5:][::-1]

    print("TOP 5 DOCUMENTOS MÁS SIMILARES:")
    print("="*80)

    for i, idx in enumerate(top_5_indices, 1):
        similarity_score = similarities[idx]
        document = documents[idx]

        print(f"\n{i}. Documento #{idx}")
        print(f"   Similaridad: {similarity_score:.4f} (0-1, mayor es mejor)")
        print(f"{'-'*80}")
        # Mostrar los primeros 400 caracteres
        print(document[:400])
        if len(document) > 400:
            print("   ...")
        print()

except Exception as e:
    print(f"Error: {e}")

Query: ¿Cuáles son los tratamientos para enfermedades del corazón?
--------------------------------------------------------------------------------

TOP 5 DOCUMENTOS MÁS SIMILARES:

1. Documento #60
   Similaridad: 0.2451 (0-1, mayor es mejor)
--------------------------------------------------------------------------------

I'd like this too... maybe you should post an answer key after a while?

Nanci


2. Documento #96
   Similaridad: 0.2392 (0-1, mayor es mejor)
--------------------------------------------------------------------------------



3. Documento #16
   Similaridad: 0.2392 (0-1, mayor es mejor)
--------------------------------------------------------------------------------





4. Documento #4
   Similaridad: 0.2392 (0-1, mayor es mejor)
--------------------------------------------------------------------------------



5. Documento #21
   Similaridad: 0.2344 (0-1, mayor es mejor)
--------------------------------------------------------------------------------

Could you 